# Forex Trading Strategy Analysis

This notebook analyzes EUR/USD Forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [11]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import sys
import importlib

# Force reload of utils module to prevent caching
if 'utils' in sys.modules:
    del sys.modules['utils']

# Enable automatic reloading of modules
%load_ext autoreload
%autoreload 2

# Import utility functions (will be freshly loaded)
import utils
from utils import (
    load_and_clean_data, 
    calculate_profitable_trades, 
    analyze_entry_timing, 
    display_profitable_strategies,
    Strategy,
    create_strategy_library,
    evaluate_all_strategies,
    get_top_strategies,
    get_top_strategies_by_edge,
    analyze_pullback_profitability
)

# Force reload utils to ensure latest version
importlib.reload(utils)

# Load the data
df = load_and_clean_data()

# Display profitable trades analysis
display(HTML("""
    <h2>Profitable Trades</h2>
    <p>What are simple trading ideas that are profitable?</p>
"""))
display(calculate_profitable_trades(df))
display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>CH trades are out of question, they're not profitable.</li>
        <li>3 pips extra is harming the strategy, while 1 pip can improve it slightly.</li>
        <li>Overall any filtration is harmful.</li>
    </ul>
"""))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Data,1:1 RRR,1:2 RRR,1:3 RRR
0,With Extra 1 pip,42.9%,25.5%,18.4%
1,With Extra 2 pips,42.9%,27.6%,14.3%
2,Total,38.8%,24.5%,20.4%
3,With Extra 3 pips,36.7%,22.4%,13.3%
4,Just BOS Trades,25.5%,17.3%,15.3%
5,With EMA Direction,23.5%,14.3%,11.2%
6,With EMA + BOS,18.4%,12.2%,10.2%
7,Against EMA Direction,15.3%,10.2%,9.2%
8,Just CH Trades,13.3%,7.1%,5.1%
9,With EMA + CH,5.1%,2.0%,1.0%


In [12]:
# Display entry timing analysis using the imported function
display(HTML("""
    <h2>When To Enter</h2>
    <p>Following market structure, price taps order block. This is when the signal is received. Now - when to enter the trade?</p>
"""))
display(analyze_entry_timing(df))
display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>Very interesting that here, my strategy with 1M confirmation candle is the worse.</li>
"""))
display(HTML("""
    <p><b>Open Questions</b></p>
    <ul>
        <li>That 5M Stop entry makes a lot of sense as all profitable strategies would have it, but in reality somehow it performs the worse. Why?</li>
        <li>Why does 1M confirmation candle entry outperforms all other entries in the next setups analysis?</li>
    </ul>
"""))

,Idea,Notation,Win Rate,With Extra,With Extra & 1:3 RRR
0,5M Stop,31W - 32L,49.2%,55.6%,28.6%
1,5M Breakout,30W - 35L,46.2%,52.3%,30.8%
2,5M Confirmation Candle,31W - 40L,43.7%,49.3%,28.2%
3,1M Confirmation Candle,38W - 60L,38.8%,42.9%,22.4%


In [13]:
# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

# Add all strategies from the library
strategies.extend(create_strategy_library())

# Evaluate all strategies
strategy_results = evaluate_all_strategies(df, strategies)

# Display top performing strategies for each RRR - sorted by Outcome
display(HTML("<h2>🏆 TOP Performing Strategies (Sorted by Outcome)</h2>"))

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies</h3>"))
    
    # Get and display top strategies
    top_df = get_top_strategies(strategy_results, rrr_column)
    top_df = top_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px', 'font-weight': 'bold'}
    ).set_properties(
        subset=['Outcome'],
        **{'color': 'green', 'font-weight': 'bold'}
    )
    
    display(styled_df)
    print()  # Add spacing

# Display top performing strategies for each RRR - sorted by Edge
display(HTML("<h2>🎯 TOP Performing Strategies (Sorted by Edge)</h2>"))

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies by Edge</h3>"))
    
    # Get and display top strategies sorted by edge
    top_df = get_top_strategies_by_edge(strategy_results, rrr_column)
    top_df = top_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px'}
    ).set_properties(
        subset=['Edge'],
        **{'color': 'green'}
    )
    
    display(styled_df)
    print()  # Add spacing

display(HTML("""
    <p><b>Summary</b></p>
    <ul>
        <li>EMA + BOS is on 5th place. So anything above it should be tradeable</li>
        <li>According this list, trading all BOS trades would simplify things a lot. Lets try that out!</li>
    </ul>
"""))

,Top 1:1 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,30M Trend + BOS,1M CC,40,23,17,57.5%,7.5%,6R
1,30M Trend + SL > 5 pips,1M CC,18,12,6,66.7%,16.7%,6R
2,30M Trend + 5 < SL < 15,1M CC,18,12,6,66.7%,16.7%,6R
3,30M Trend + BOS + SL < 10,1M CC,40,23,17,57.5%,7.5%,6R
4,30M Trend + SL > 5 pips,5M CC,18,12,6,66.7%,16.7%,6R
5,30M Trend + 5 < SL < 15,5M CC,18,12,6,66.7%,16.7%,6R
6,30M Trend + SL > 5 pips,5M Breakout,18,12,6,66.7%,16.7%,6R
7,30M Trend + 5 < SL < 15,5M Breakout,18,12,6,66.7%,16.7%,6R
8,BOS,1M CC,61,33,28,54.1%,4.1%,5R
9,EMA + Opposite Trade Direction,1M CC,38,21,17,55.3%,5.3%,4R


,Top 1:2 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,30M Trend + BOS,1M CC,40,18,22,45.0%,11.7%,14R
1,30M Trend + BOS + SL < 10,1M CC,40,18,22,45.0%,11.7%,14R
2,30M Trend + BOS,5M CC,40,18,22,45.0%,11.7%,14R
3,30M Trend + BOS + SL < 10,5M CC,40,18,22,45.0%,11.7%,14R
4,30M Trend + BOS,5M Stop,40,18,22,45.0%,11.7%,14R
5,30M Trend + BOS + SL < 10,5M Stop,40,18,22,45.0%,11.7%,14R
6,BOS,1M CC,61,24,37,39.3%,6.0%,11R
7,BOS,5M Stop,61,24,37,39.3%,6.0%,11R
8,30M Trend + BOS,5M Breakout,40,17,23,42.5%,9.2%,11R
9,30M Trend + SL < 10 pips,5M Breakout,58,23,35,39.7%,6.4%,11R


,Top 1:3 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,1M CC,61,22,39,36.1%,11.1%,27R
1,30M Trend + BOS,1M CC,40,16,24,40.0%,15.0%,24R
2,30M Trend + BOS + SL < 10,1M CC,40,16,24,40.0%,15.0%,24R
3,BOS,5M Breakout,61,20,41,32.8%,7.8%,19R
4,30M Trend + SL < 10 pips,1M CC,58,19,39,32.8%,7.8%,18R
5,30M Trend + EMA + BOS,1M CC,35,13,22,37.1%,12.1%,17R
6,30M Trend + EMA + BOS + SL < 10,1M CC,35,13,22,37.1%,12.1%,17R
7,With 30M Trend,1M CC,60,19,41,31.7%,6.7%,16R
8,30M Trend + SL < 15 pips,1M CC,60,19,41,31.7%,6.7%,16R
9,30M Trend + BOS,5M CC,40,14,26,35.0%,10.0%,16R


,Top 1:1 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,30M Trend + SL > 5 pips,1M CC,18,12,6,66.7%,16.7%,6R
1,30M Trend + 5 < SL < 15,1M CC,18,12,6,66.7%,16.7%,6R
2,30M Trend + SL > 5 pips,5M CC,18,12,6,66.7%,16.7%,6R
3,30M Trend + 5 < SL < 15,5M CC,18,12,6,66.7%,16.7%,6R
4,30M Trend + SL > 5 pips,5M Breakout,18,12,6,66.7%,16.7%,6R
5,30M Trend + 5 < SL < 15,5M Breakout,18,12,6,66.7%,16.7%,6R
6,30M Trend + SL > 5 pips,5M Stop,18,11,7,61.1%,11.1%,4R
7,30M Trend + 5 < SL < 15,5M Stop,18,11,7,61.1%,11.1%,4R
8,Aggressive: SL >= 7 pips,1M CC,17,10,7,58.8%,8.8%,3R
9,30M Trend + News > 2hrs,1M CC,12,7,5,58.3%,8.3%,2R


,Top 1:2 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,8,4,4,50.0%,16.7%,4R
1,30M Trend + News > 2hrs,1M CC,12,6,6,50.0%,16.7%,6R
2,30M Trend + News > 2hrs,5M CC,12,6,6,50.0%,16.7%,6R
3,30M Trend + News > 2hrs,5M Stop,12,6,6,50.0%,16.7%,6R
4,30M Trend + SL > 5 pips,5M Breakout,18,9,9,50.0%,16.7%,9R
5,30M Trend + 5 < SL < 15,5M Breakout,18,9,9,50.0%,16.7%,9R
6,30M Trend + News > 2hrs,5M Breakout,12,6,6,50.0%,16.7%,6R
7,News in 2+ Hours,1M CC,26,12,14,46.2%,12.9%,10R
8,30M Trend + BOS,1M CC,40,18,22,45.0%,11.7%,14R
9,30M Trend + BOS + SL < 10,1M CC,40,18,22,45.0%,11.7%,14R


,Top 1:3 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS + Conservative SL,1M CC,8,4,4,50.0%,25.0%,8R
1,30M Trend + News > 2hrs,1M CC,12,5,7,41.7%,16.7%,8R
2,30M Trend + News > 2hrs,5M CC,12,5,7,41.7%,16.7%,8R
3,30M Trend + News > 2hrs,5M Breakout,12,5,7,41.7%,16.7%,8R
4,30M Trend + BOS,1M CC,40,16,24,40.0%,15.0%,24R
5,30M Trend + BOS + SL < 10,1M CC,40,16,24,40.0%,15.0%,24R
6,30M Trend + SL > 5 pips,5M CC,18,7,11,38.9%,13.9%,10R
7,30M Trend + 5 < SL < 15,5M CC,18,7,11,38.9%,13.9%,10R
8,News in 2+ Hours,5M CC,26,10,16,38.5%,13.5%,14R
9,BOS + Conservative SL,5M Stop,8,3,5,37.5%,12.5%,4R


In [14]:
# Pullback Analysis
display(HTML("<h2>Pullback Impact on Profitability</h2>"))
display(HTML("<p>How does pullback size affect trade profitability? Analyzing trades where TP > SL.</p>"))

# Display pullback profitability analysis
pullback_analysis = analyze_pullback_profitability(df)
display(pullback_analysis)

# Add summary
display(HTML("""
<p><b>Summary</b></p>
<ul>
    <li>According to this analysis, pullback of 0.5 pip should not harm the strategy but could dramatically reduce the the path to the TP.</li>
</ul>
"""))

,Pullback,All Trades,Profitable Trades,Percentage
0,All (No Filter),98,50,51.0%
1,0.5 pips,91,45,49.5%
2,1.0 pip,84,38,45.2%
3,1.5 pips,79,34,43.0%
4,2.0 pips,66,28,42.4%
5,2.5 pips,58,21,36.2%
6,3.0 pips,52,18,34.6%


In [15]:
# Display only profitable strategies
display_profitable_strategies(strategy_results)

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,98,98,98
1,Wins,50,33,28
2,Losses,48,65,70
3,Win Rate,51.0%,33.7%,28.6%
4,Edge,1.0%,0.4%,3.6%
5,Outcome,2R,1R,14R
6,Entry,1M CC,1M CC,1M CC


,EMA + Opposite Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,38,38,38
1,Wins,21,14,13
2,Losses,17,24,25
3,Win Rate,55.3%,36.8%,34.2%
4,Edge,5.3%,3.5%,9.2%
5,Outcome,4R,4R,14R
6,Entry,1M CC,1M CC,1M CC


,BOS,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,61,61,61
1,Wins,33,24,22
2,Losses,28,37,39
3,Win Rate,54.1%,39.3%,36.1%
4,Edge,4.1%,6.0%,11.1%
5,Outcome,5R,11R,27R
6,Entry,1M CC,1M CC,1M CC


,Moderate Risk: SL 3-6 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,53,53,53
1,Wins,27,16,13
2,Losses,26,37,40
3,Win Rate,50.9%,30.2%,24.5%
4,Edge,0.9%,-3.1%,-0.5%
5,Outcome,1R,-5R,-1R
6,Entry,1M CC,1M CC,1M CC


,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,17,17,17
1,Wins,10,5,4
2,Losses,7,12,13
3,Win Rate,58.8%,29.4%,23.5%
4,Edge,8.8%,-3.9%,-1.5%
5,Outcome,3R,-2R,-1R
6,Entry,1M CC,1M CC,1M CC


...

,30M Trend + News > 2hrs,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,12,12,12
1,Wins,7,6,4
2,Losses,5,6,8
3,Win Rate,58.3%,50.0%,33.3%
4,Edge,8.3%,16.7%,8.3%
5,Outcome,2R,6R,4R
6,Entry,5M Stop,5M Stop,5M Stop


,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,17,17,17
1,Wins,9,6,4
2,Losses,8,11,13
3,Win Rate,52.9%,35.3%,23.5%
4,Edge,2.9%,2.0%,-1.5%
5,Outcome,1R,1R,-1R
6,Entry,5M Breakout,5M Breakout,5M Breakout


,30M Trend + SL > 5 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,18,18,18
1,Wins,12,9,6
2,Losses,6,9,12
3,Win Rate,66.7%,50.0%,33.3%
4,Edge,16.7%,16.7%,8.3%
5,Outcome,6R,9R,6R
6,Entry,5M Breakout,5M Breakout,5M Breakout


,30M Trend + 5 < SL < 15,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,18,18,18
1,Wins,12,9,6
2,Losses,6,9,12
3,Win Rate,66.7%,50.0%,33.3%
4,Edge,16.7%,16.7%,8.3%
5,Outcome,6R,9R,6R
6,Entry,5M Breakout,5M Breakout,5M Breakout


,30M Trend + News > 2hrs,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,12,12,12
1,Wins,7,6,5
2,Losses,5,6,7
3,Win Rate,58.3%,50.0%,41.7%
4,Edge,8.3%,16.7%,16.7%
5,Outcome,2R,6R,8R
6,Entry,5M Breakout,5M Breakout,5M Breakout
